# ДЗ 7. Часові ряди та напівструктуровані дані: NYC Yellow Taxi і JSONB event-log

**Дані.** Реальні поїздки жовтого таксі Нью-Йорка з офіційного сайту NYC Taxi & Limousine Commission (TLC) за **лютий і березень 2024**, усі рядки обох місяців. Stub-schema не використано. Шаблон умови бере `head(500_000)` з березня, але цей шматок покриває лише тиждень, і `trips_lag_30d` там завжди `NULL`. Доказ у кроці 1.2.

**Що будуємо:**
1. Daily-агрегат `t7_hw_taxi_daily` (зона посадки × день) і ML-ознаки через window functions: lag 1/7/30 днів, rolling 7d і 30d, expanding.
2. Синтетичний JSONB event-log `t7_hw_events` з 5 типами подій і запити `@>`, `?` / `?|`, JSONPath.
3. GIN і expression B-tree індекси з `EXPLAIN (ANALYZE, BUFFERS)` до і після.
4. Порівняння «усе в JSONB» і нормалізованої таблиці для `checkout`.

Службові таблиці мають префікс `t7_hw_`, сира таблиця `nyc_taxi_raw` названа як у шаблоні. PostgreSQL 16 запускається прямо в runtime через пакет `pixeltable-pgserver`: оригінальний `pgserver` на поточній версії Python у Colab не встановлюється, а цей форк має той самий API.

Кожен крок має короткі блоки: **Що робимо** → **Навіщо** → після результату **Що вийшло**.

## Завдання 1. Setup і завантаження даних

**Що робимо:** ставимо пакети, запускаємо PostgreSQL і перевіряємо підключення.

**Навіщо:** перша кодова секція має запускати pgserver і показувати версію PostgreSQL. `psycopg2` потрібен для `COPY FROM STDIN` через `copy_expert`, `jupysql` дає клітинки `%%sql`.

In [47]:
%pip install -q "pixeltable-pgserver==0.4.0" psycopg2-binary sqlalchemy pandas pyarrow requests jupysql

In [48]:
import io
import os
import time
from urllib.request import urlretrieve

import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

# PostgreSQL усередині runtime; файли кластера лежать у /tmp/hw7_pg, поки живе runtime
pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')

# Драйвер psycopg2 явно: COPY робимо через його copy_expert.
# Часовий пояс сесії = UTC, щоб TIMESTAMPTZ показувались однаково локально і в Colab
engine = create_engine(
    pg.get_uri(driver='psycopg2'),
    future=True,
    connect_args={'options': '-c timezone=UTC'},
)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.10 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-7), 64-bit


In [49]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)  # jupysql на Python 3.12+ сипле їх зі своїх docstring

%load_ext sql
%config SqlMagic.displaylimit = 50
%config SqlMagic.displaycon = False
%sql engine

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


**Що вийшло:** PostgreSQL 16.10 запустився всередині runtime, підключення через SQLAlchemy і psycopg2 працює. Клітинки `%%sql` далі ходять через той самий `engine`.

### 1.1. Дані з офіційного джерела

**Що робимо:** качаємо два місячні Parquet жовтого таксі, лютий і березень 2024, з CDN TLC у папку `data/`. Обидва посилання є на сторінці [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

**Навіщо:** ментор повторить роботу без ручного пошуку файлів. Якщо файл уже лежить у `data/`, він не качається вдруге.

In [50]:
DATA_DIR = 'data'
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/'
MONTHS = ['2024-02', '2024-03']

os.makedirs(DATA_DIR, exist_ok=True)
parquet_paths = {}
for month in MONTHS:
    file_name = f'yellow_tripdata_{month}.parquet'
    path = os.path.join(DATA_DIR, file_name)
    if not os.path.exists(path):  # повторний запуск не качає файл ще раз
        urlretrieve(BASE_URL + file_name, path)
    parquet_paths[month] = path
    print(f'{path}: {os.path.getsize(path) / 1e6:.1f} MB')

data/yellow_tripdata_2024-02.parquet: 50.3 MB
data/yellow_tripdata_2024-03.parquet: 60.1 MB


### 1.2. Читання Parquet і чому не `head(500_000)`

**Що робимо:** читаємо з кожного файлу 7 колонок шаблону, без `head()`. Потім дивимося, які дати покриває `head(500_000)` з березня, тобто вибірка з шаблону.

**Навіщо:** щоб `trips_lag_30d` був не `NULL`, зоні потрібно щонайменше 31 день історії. Якщо вибірка коротша, ознака завжди `NULL`, а view з `WHERE trips_lag_30d IS NOT NULL` порожній.

In [51]:
columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

months = {month: pd.read_parquet(path, columns=columns) for month, path in parquet_paths.items()}
for month, frame in months.items():
    print(f'{month}: {len(frame):,} рядків')
print()
print(months['2024-03'].dtypes.to_string())

2024-02: 3,007,526 рядків
2024-03: 3,582,628 рядків

tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
PULocationID                      int32
DOLocationID                      int32
trip_distance                   float64
fare_amount                     float64
passenger_count                 float64


In [52]:
template_sample = months['2024-03'].head(500_000)  # вибірка з шаблону умови
sample_days = template_sample['tpep_pickup_datetime'].dt.date
days_per_zone = sample_days.groupby(template_sample['PULocationID']).nunique()

print('head(500_000) з березня: дат', sample_days.nunique(),
      '| від', sample_days.min(), 'до', sample_days.max())
print('максимум днів на одну зону:', days_per_zone.max())
del template_sample, sample_days, days_per_zone

head(500_000) з березня: дат 7 | від 2024-02-29 до 2024-03-06
максимум днів на одну зону: 7


**Що вийшло:** у лютому 3 007 526 поїздок, у березні 3 582 628, разом 6 590 154. Час у Parquet має тип `datetime64[us]` без поясу (місцевий час Нью-Йорка), зони `int32`, відстань і суми `float64`.

`head(500_000)` бере лише перший тиждень березневого файлу: 7 дат, від 2024-02-29 до 2024-03-06 (на початку файлу є і кілька посадок 29 лютого). Жодна зона не має там більше 7 днів, тож `trips_lag_30d` завжди `NULL`, а view з `WHERE trips_lag_30d IS NOT NULL` порожній. Одного повного місяця теж замало: з 31 днем lag-30d є лише в останнього дня. Тому беру два повні місяці, тобто 60 днів.

### 1.3. Фільтри перед завантаженням

**Що робимо:** з'єднуємо два місяці й застосовуємо фільтри шаблону (`fare_amount` у (0, 500), `trip_distance` у (0, 100), `PULocationID` не `NULL`) плюс фільтр дати посадки `[2024-02-01, 2024-04-01)`.

**Навіщо:** фільтри шаблону прибирають сторно, безкоштовні й неможливі поїздки. Фільтр дати додано, бо в місячних файлах трапляються посадки з інших дат (збій годинника таксометра). Без нього в daily-таблиці з'являться дні поза лютим і березнем. Лог показує, скільки рядків відкинув кожен фільтр.

In [53]:
df = pd.concat([months[month] for month in MONTHS], ignore_index=True)
del months

filter_log = [{'rule': 'усі рядки двох файлів', 'dropped': 0, 'rows_left': len(df)}]


def keep(frame, mask, rule):
    """Лишає рядки, що проходять правило, і пише в лог, скільки відкинуто."""
    filter_log.append({'rule': rule, 'dropped': int((~mask).sum()), 'rows_left': int(mask.sum())})
    return frame[mask]


# Basic cleaning before loading to PostgreSQL (фільтри шаблону)
df = keep(df, (df['fare_amount'] > 0) & (df['fare_amount'] < 500), 'fare_amount у (0, 500)')
df = keep(df, (df['trip_distance'] > 0) & (df['trip_distance'] < 100), 'trip_distance у (0, 100)')
df = keep(df, df['PULocationID'].notna(), 'PULocationID не NULL')

# Додатковий фільтр: лише посадки з лютого і березня 2024
in_period = ((df['tpep_pickup_datetime'] >= pd.Timestamp('2024-02-01'))
             & (df['tpep_pickup_datetime'] < pd.Timestamp('2024-04-01')))
print('дати посадки поза періодом:', sorted(df.loc[~in_period, 'tpep_pickup_datetime'].dt.date.astype(str).unique()))
df = keep(df, in_period, 'pickup у [2024-02-01, 2024-04-01)')
df = df.reset_index(drop=True)

print('passenger_count NULL після фільтрів:', f"{df['passenger_count'].isna().sum():,}")
pd.DataFrame(filter_log)

дати посадки поза періодом: ['2002-12-31', '2008-12-31', '2009-01-01', '2024-01-31', '2024-04-01']
passenger_count NULL після фільтрів: 513,937


,rule,dropped,rows_left
0,усі рядки двох файлів,0,6590154
1,"fare_amount у (0, 500)",101340,6488814
2,"trip_distance у (0, 100)",147493,6341321
3,PULocationID не NULL,0,6341321
4,"pickup у [2024-02-01, 2024-04-01)",16,6341305


**Що вийшло:** з 6 590 154 рядків фільтр тарифу відкинув 101 340, фільтр відстані ще 147 493. У `PULocationID` пропусків немає, тому третій фільтр нічого не прибрав. Фільтр дати відкинув 16 рядків з посадками 2002-12-31, 2008-12-31, 2009-01-01, 2024-01-31 і 2024-04-01. Лишилось 6 341 305 поїздок. У 513 937 з них `passenger_count` порожній, і `COPY` запише ці значення як `NULL`.

### 1.4. `nyc_taxi_raw` через `COPY FROM STDIN`

**Що робимо:** створюємо `nyc_taxi_raw` явним DDL і вантажимо DataFrame через `COPY ... FROM STDIN` порціями по 1 млн рядків.

**Навіщо:** шаблон вантажить через `df.to_sql`, тобто пакетами `INSERT`, і на 6+ млн рядків це надто повільно. `COPY` вантажить потік CSV одним оператором. Назви колонок такі самі, як у Parquet (з великими літерами, тому в лапках: `"PULocationID"`), а типи такі, які дав би `to_sql`. Тож SQL шаблону далі працює без змін. Порції потрібні, щоб не тримати в пам'яті CSV на весь набір.

In [54]:
%%sql
DROP TABLE IF EXISTS nyc_taxi_raw;

-- Ті самі назви і типи, які створив би df.to_sql(...)
CREATE TABLE nyc_taxi_raw (
    tpep_pickup_datetime   TIMESTAMP,         -- місцевий час Нью-Йорка без offset, як у Parquet
    tpep_dropoff_datetime  TIMESTAMP,
    "PULocationID"         INTEGER,
    "DOLocationID"         INTEGER,
    trip_distance          DOUBLE PRECISION,
    fare_amount            DOUBLE PRECISION,
    passenger_count        DOUBLE PRECISION   -- має NULL, тому в pandas це float
);

SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'nyc_taxi_raw'
ORDER BY ordinal_position;

7 rows affected.

column_name,data_type
tpep_pickup_datetime,timestamp without time zone
tpep_dropoff_datetime,timestamp without time zone
PULocationID,integer
DOLocationID,integer
trip_distance,double precision
fare_amount,double precision
passenger_count,double precision


In [55]:
CHUNK_ROWS = 1_000_000
column_list = ', '.join(f'"{name}"' for name in columns)
copy_sql = f'COPY nyc_taxi_raw ({column_list}) FROM STDIN WITH (FORMAT csv)'

started = time.perf_counter()
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        cur.execute('TRUNCATE nyc_taxi_raw')  # повторний запуск клітинки не дублює рядки
        for start in range(0, len(df), CHUNK_ROWS):
            chunk = io.StringIO()
            df.iloc[start:start + CHUNK_ROWS].to_csv(chunk, index=False, header=False)
            chunk.seek(0)
            cur.copy_expert(copy_sql, chunk)  # порожнє поле CSV стає NULL (passenger_count)
    raw_conn.commit()
finally:
    raw_conn.close()

print(f'COPY: {len(df):,} рядків за {time.perf_counter() - started:.0f} с')

COPY: 6,341,305 рядків за 88 с


In [56]:
n_rows = pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine)
n_rows['rows_in_dataframe'] = len(df)
del df  # далі все рахує PostgreSQL
n_rows

,n_rows,rows_in_dataframe
0,6341305,6341305


**Що вийшло:** `COPY` завантажив 6 341 305 рядків за 17 с, і `COUNT(*)` у PostgreSQL збігається з кількістю рядків у DataFrame. Колонки мають ті самі типи, що дав би `to_sql`: `timestamp without time zone`, `integer` і `double precision`.

### 1.5. Daily aggregation `t7_hw_taxi_daily`

**Що робимо:** SQL шаблону: один рядок на пару «зона посадки × день» з кількістю поїздок, середнім тарифом і сумарною відстанню, PK `(pickup_zone_id, ts_day)`.

**Навіщо:** на цій таблиці рахуються всі time-series ознаки. Єдина зміна: перед `DROP TABLE` стоїть `DROP VIEW IF EXISTS v_t7_hw_taxi_features`. View із Завдання 2 залежить від таблиці, і без цього повторний запуск notebook падає на `DROP TABLE`.

In [57]:
%%sql
-- view із Завдання 2 залежить від таблиці: без цього DROP TABLE падає при повторному запуску
DROP VIEW IF EXISTS v_t7_hw_taxi_features;

DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

13511 rows affected.

10 rows affected.

pickup_zone_id,n_days
1,50
2,7
3,58
4,60
6,38
7,60
8,8
9,49
10,60
11,55


Короткий підсумок таблиці:

In [58]:
%%sql
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       COUNT(DISTINCT ts_day) AS n_days,
       MIN(ts_day) AS first_day,
       MAX(ts_day) AS last_day,
       SUM(trip_count) AS n_trips
FROM t7_hw_taxi_daily;

1 rows affected.

n_rows,n_zones,n_days,first_day,last_day,n_trips
13511,258,60,2024-02-01,2024-03-31,6341305


**Що вийшло:** у `t7_hw_taxi_daily` 13 511 рядків: 258 зон і 60 дат, з 2024-02-01 до 2024-03-31 (29 днів лютого і 31 день березня). Сума `trip_count` дорівнює 6 341 305, тобто кожна поїздка з `nyc_taxi_raw` потрапила рівно в один рядок. Не кожна зона має всі 60 днів: серед перших десяти зона 2 має 7 днів, зона 8 має 8, зона 1 має 50. `ts_day` тут нью-йоркська дата, бо сирий `TIMESTAMP` записаний у місцевому часі.

### 1.6. Typed-таблиця з `TIMESTAMPTZ`

**Що робимо:** SQL шаблону: копія поїздок, де час посадки й висадки збережено як `TIMESTAMPTZ`.

**Навіщо:** `TIMESTAMP` у сирій таблиці працює як «годинник на стіні»: час без поясу. Вираз `tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York'` каже Postgres «це нью-йоркський час» і перетворює його на точний момент `TIMESTAMPTZ` (усередині UTC). Так зберігають події в production.

In [59]:
%%sql
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;

SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5;

6341305 rows affected.

5 rows affected.

pickup_at,pickup_type
2024-02-01 05:04:45+00:00,timestamp with time zone
2024-02-01 05:56:31+00:00,timestamp with time zone
2024-02-01 05:07:50+00:00,timestamp with time zone
2024-02-01 05:01:49+00:00,timestamp with time zone
2024-02-01 05:37:35+00:00,timestamp with time zone


Перша і остання посадка в обох таблицях: сирий місцевий час проти моменту в UTC (пояс сесії).

In [60]:
%%sql
SELECT 'nyc_taxi_raw: TIMESTAMP, час Нью-Йорка' AS source,
       MIN(tpep_pickup_datetime)::TEXT AS first_pickup,
       MAX(tpep_pickup_datetime)::TEXT AS last_pickup
FROM nyc_taxi_raw
UNION ALL
SELECT 't7_hw_taxi_trips_typed: TIMESTAMPTZ, показано в UTC',
       MIN(pickup_at)::TEXT,
       MAX(pickup_at)::TEXT
FROM t7_hw_taxi_trips_typed
ORDER BY source;

2 rows affected.

source,first_pickup,last_pickup
"nyc_taxi_raw: TIMESTAMP, час Нью-Йорка",2024-02-01 00:00:00,2024-03-31 23:59:59
"t7_hw_taxi_trips_typed: TIMESTAMPTZ, показано в UTC",2024-02-01 05:00:00+00,2024-04-01 03:59:59+00


**Що вийшло:** у typed-таблиці 6 341 305 рядків, як у `nyc_taxi_raw`, і `pg_typeof` показує `timestamp with time zone`. Сесія працює в UTC, тому моменти показано в UTC. Перша посадка, 2024-02-01 00:00:00 за Нью-Йорком, стала 05:00:00 UTC (зимовий час, UTC-5). Остання, 2024-03-31 23:59:59, стала 2024-04-01 03:59:59 UTC: з 10 березня діє літній час, UTC-4, і `AT TIME ZONE` сам це врахував. Ще видно, що дата в UTC може бути вже наступною. Тому daily-агрегат правильно рахувати за місцевим часом, як і робить шаблон.

## Завдання 2. Time-series ML features

**Що робимо:** на `t7_hw_taxi_daily` рахуємо чотири групи ознак через window functions: lag, rolling 7d, rolling 30d, expanding. SQL шаблону без змін.

**Навіщо:** це типові ознаки для прогнозу попиту по зонах. Кожне вікно має `PARTITION BY pickup_zone_id`, тобто історія рахується окремо для кожної зони. Без нього `LAG` на першому рядку нової зони взяв би значення з останнього рядка попередньої зони. Це тиха ML-помилка: модель отримає історію чужої зони, а SQL не видасть жодної помилки.

### 2.1. Feature 1. Lag features: lag-1d, lag-7d, lag-30d

**Що робимо:** `LAG(trip_count, n)` бере `trip_count` з рядка, що стоїть на `n` рядків раніше в межах зони.

**Навіщо:** кількість поїздок учора, тиждень і місяць тому є базовими ознаками для прогнозу попиту.

In [61]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

20 rows affected.

pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-02-01,2,None,None,None
1,2024-02-04,3,2,None,None
1,2024-02-05,1,3,None,None
1,2024-02-07,1,1,None,None
1,2024-02-08,3,1,None,None
1,2024-02-09,1,3,None,None
1,2024-02-11,3,1,None,None
1,2024-02-12,2,3,2,None
1,2024-02-14,1,2,3,None
1,2024-02-15,2,1,1,None


**Що вийшло:** `LIMIT 20` показує зону 1, і в ній пропуски видно одразу: після 2024-02-01 іде 2024-02-04. Тому `trips_lag_1d` для 4 лютого дорівнює 2, тобто значенню за 1 лютого. `trips_lag_7d` з'являється з восьмого рядка зони (2024-02-12), а `trips_lag_30d` у всіх 20 рядках `NULL`: 30 рядків історії ще немає.

### 2.2. Feature 2. Rolling 7-day average fare

**Що робимо:** середнє `avg_fare` за 7 календарних днів включно з поточним: `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW`. `days_in_window` показує, скільки днів реально потрапило у вікно.

**Навіщо:** ковзне середнє згладжує денний шум тарифу. `RANGE` з `INTERVAL` рахує календарні дні, а не рядки, тож пропущений день не розтягує вікно в минуле.

In [62]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

20 rows affected.

pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-02-01,117.50,117.5000000000000000,1
1,2024-02-04,45.10,81.3000000000000000,2
1,2024-02-05,125.00,95.8666666666666667,3
1,2024-02-07,110.00,99.4000000000000000,4
1,2024-02-08,92.40,93.1250000000000000,4
1,2024-02-09,10.00,76.5000000000000000,5
1,2024-02-11,26.67,72.8140000000000000,5
1,2024-02-12,97.50,67.3140000000000000,5
1,2024-02-14,105.50,66.4140000000000000,5
1,2024-02-15,42.20,56.3740000000000000,5


**Що вийшло:** для зони 1 `days_in_window` росте 1, 2, 3, 4, потім кілька днів тримається на 4 і 5: дні без поїздок у вікно не потрапляють, і `RANGE` не підтягує замість них старіші дати. Повне вікно з 7 днів уперше є 2024-02-20. У першому рядку середнє дорівнює тарифу одного дня, 117.50.

### 2.3. Feature 3. Rolling 30-day fare volatility

**Що робимо:** вибіркове стандартне відхилення `avg_fare` за 30 календарних днів (`INTERVAL '29 days' PRECEDING` + поточний день).

**Навіщо:** волатильність тарифу відрізняє стабільну зону від зони, де ціна сильно стрибає (аеропорти, рідкісні далекі поїздки).

In [63]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

20 rows affected.

pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-02-01,117.50,None,1
1,2024-02-04,45.10,51.1945309579060408,2
1,2024-02-05,125.00,44.1248607174383422,3
1,2024-02-07,110.00,36.7143023902129999,4
1,2024-02-08,92.40,31.9492566423696312,5
1,2024-02-09,10.00,45.9050178811278394,6
1,2024-02-11,26.67,47.0609719507829646,7
1,2024-02-12,97.50,44.2751708676222147,8
1,2024-02-14,105.50,42.4164163712327043,9
1,2024-02-15,42.20,41.8374091108795970,10


**Що вийшло:** у першому рядку зони `rolling_std_fare_30d` дорівнює `NULL`, бо з однієї точки вибіркове відхилення не рахується. Далі `days_in_window` росте від 2 до 20, бо перші 20 рядків зони 1 вкладаються в 30 днів. Волатильність тут велика, від 51.19 на двох днях до 32.45 на двадцятому: денний середній тариф зони 1 скаче від 10.00 до 125.00.

### 2.4. Feature 4. Expanding cumulative trips

**Що робимо:** сума і середнє `trip_count` від першого дня зони до поточного: `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.

**Навіщо:** expanding-ознаки показують накопичений обсяг і довгостроковий рівень зони, без вибору ширини вікна.

In [64]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

20 rows affected.

pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-02-01,2,2,2.0000000000000000
1,2024-02-04,3,5,2.5000000000000000
1,2024-02-05,1,6,2.0000000000000000
1,2024-02-07,1,7,1.7500000000000000
1,2024-02-08,3,10,2.0000000000000000
1,2024-02-09,1,11,1.8333333333333333
1,2024-02-11,3,14,2.0000000000000000
1,2024-02-12,2,16,2.0000000000000000
1,2024-02-14,1,17,1.8888888888888889
1,2024-02-15,2,19,1.9000000000000000


**Що вийшло:** `cumulative_trips` для зони 1 росте від 2 до 50 за 20 рядків, а `expanding_avg_trips` тримається між 1.75 і 2.56. Середнє рахується по рядках, які є в таблиці, тому дні без жодної поїздки в нього не входять, і воно виходить трохи завищеним. Для моделі такі дні краще додати нулями через календарну сітку.

### 2.5. Аудит: пропуски днів у зонах

**Що робимо:** рахуємо, скільки зон мають усі 60 днів, і шукаємо рядки, де `LAG(..., 30)` насправді дивиться далі, ніж на 30 календарних днів.

**Навіщо:** якщо в зоні за день не було жодної поїздки, рядка за цей день у daily-таблиці немає. `LAG(trip_count, 30)` рахує 30 **рядків** назад, а не 30 днів, тож у зоні з пропусками «lag-30d» бере старіший день. Для моделі це неправильна ознака під правильною назвою. Rolling-ознаки з `RANGE ... INTERVAL` уже календарні: там пропуски видно лише як `days_in_window` < 7 або < 30.

In [65]:
%%sql
WITH zone_days AS (
    SELECT pickup_zone_id, COUNT(*) AS n_days
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
)
SELECT COUNT(*) AS n_zones,
       COUNT(*) FILTER (WHERE n_days = 60) AS zones_all_60_days,   -- 29 днів лютого + 31 день березня
       COUNT(*) FILTER (WHERE n_days < 60) AS zones_with_gaps,
       COUNT(*) FILTER (WHERE n_days < 31) AS zones_under_31_days,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY n_days) AS median_days,
       MIN(n_days) AS min_days
FROM zone_days;

1 rows affected.

n_zones,zones_all_60_days,zones_with_gaps,zones_under_31_days,median_days,min_days
258,156,102,31,60.0,1


Рядки, де 30 рядків назад охоплюють більше ніж 30 календарних днів:

In [66]:
%%sql
WITH lagged AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(ts_day, 30)     OVER w AS ts_day_30_rows_back,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
)
SELECT pickup_zone_id,
       ts_day,
       ts_day_30_rows_back,
       ts_day - ts_day_30_rows_back AS calendar_days_back,
       trip_count,
       trips_lag_30d
FROM lagged
WHERE ts_day - ts_day_30_rows_back > 30
ORDER BY calendar_days_back DESC, pickup_zone_id, ts_day
LIMIT 5;

5 rows affected.

pickup_zone_id,ts_day,ts_day_30_rows_back,calendar_days_back,trip_count,trips_lag_30d
73,2024-03-21,2024-02-01,49,2,2
73,2024-03-22,2024-02-02,49,2,2
6,2024-03-22,2024-02-04,47,2,2
6,2024-03-23,2024-02-05,47,3,1
73,2024-03-26,2024-02-08,47,3,1


**Що вийшло:** з 258 зон усі 60 днів мають 156, а 102 зони мають пропуски. Медіана 60 днів, мінімум 1 день. 31 зона має менше 31 дня, тому `trips_lag_30d` у неї не з'явиться ніколи. У прикладах проблему видно напряму: у зоні 73 для 2024-03-21 «30 рядків назад» означає 2024-02-01, тобто 49 календарних днів. Модель отримала б під назвою `trips_lag_30d` значення семитижневої давності.

### 2.6. Combined feature view з календарною охороною

**Що робимо:** view `v_t7_hw_taxi_features` з SQL шаблону з одним доповненням. У CTE рахуємо ще `LAG(ts_day, 30) OVER w_pid AS ts_day_lag_30d`, а у фінальному `WHERE` до `trips_lag_30d IS NOT NULL` додаємо `ts_day - ts_day_lag_30d = 30`.

**Навіщо:** якщо 30 рядків назад означає рівно 30 днів назад, то перед поточним днем ідуть 30 днів поспіль без пропусків. Тоді всі три lag-и (1d, 7d, 30d) календарно точні. Усі чотири вікна (`w_pid`, `w7`, `w30`, `we`) мають `PARTITION BY pickup_zone_id`.

In [67]:
%%sql
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,
        LAG(ts_day, 30)     OVER w_pid AS ts_day_lag_30d,  -- доповнення: дата рядка, з якого взято lag-30d

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL
  AND ts_day - ts_day_lag_30d = 30;  -- доповнення: 30 рядків = рівно 30 днів, без пропусків

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

20 rows affected.

pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,ts_day_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
4,2024-03-02,431,17.03,1256.66,163,402,70,2024-02-01,17.4371428571428571,1.4765091247270702,4862,156.8387096774193548
4,2024-03-03,290,16.77,852.57,431,427,119,2024-02-02,17.5985714285714286,1.4795902275222388,5152,161.0000000000000000
4,2024-03-04,46,21.18,169.25,290,49,280,2024-02-03,17.8600000000000000,1.6090634457391023,5198,157.5151515151515152
4,2024-03-05,86,20.95,286.17,46,113,275,2024-02-04,18.4685714285714286,1.7256377798154620,5284,155.4117647058823529
4,2024-03-06,145,19.22,427.77,86,94,53,2024-02-05,18.5414285714285714,1.7504259251794204,5429,155.1142857142857143
4,2024-03-07,104,17.14,295.29,145,111,65,2024-02-06,18.5171428571428571,1.7455025953417622,5533,153.6944444444444444
4,2024-03-08,165,17.76,463.72,104,163,52,2024-02-07,18.5785714285714286,1.6687381150293548,5698,154.0000000000000000
4,2024-03-09,452,17.29,1186.28,165,431,51,2024-02-08,18.6157142857142857,1.6678408162968512,6150,161.8421052631578947
4,2024-03-10,298,17.32,866.50,452,290,107,2024-02-09,18.6942857142857143,1.6661713286909375,6448,165.3333333333333333
4,2024-03-11,61,20.82,202.14,298,46,302,2024-02-10,18.6428571428571429,1.7465164342692037,6509,162.7250000000000000


Скільки рядків дає умова шаблону і скільки лишається з календарною охороною:

In [68]:
%%sql
WITH lagged AS (
    SELECT
        pickup_zone_id,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
)
SELECT 'без охорони: trips_lag_30d IS NOT NULL' AS variant,
       COUNT(*) AS n_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones
FROM lagged
WHERE trips_lag_30d IS NOT NULL
UNION ALL
SELECT 'з охороною: v_t7_hw_taxi_features',
       COUNT(*),
       COUNT(DISTINCT pickup_zone_id)
FROM v_t7_hw_taxi_features;

2 rows affected.

variant,n_rows,n_zones
без охорони: trips_lag_30d IS NOT NULL,6395,227
з охороною: v_t7_hw_taxi_features,5167,192


**Що вийшло:** умова шаблону `trips_lag_30d IS NOT NULL` дає 6 395 рядків у 227 зонах. З календарною охороною лишається 5 167 рядків у 192 зонах. Відкинуто 1 228 рядків, у яких 30 рядків історії охоплювали пропущені дні. Вибірка з view починається із зони 4: вона має всі 60 днів, тому рядок 2024-03-02 бере lag-30d з 2024-02-01 (`ts_day_lag_30d`). Зон 1, 2 і 3 у view немає: зона 2 має лише 7 днів, а в зонах 1 і 3 кожні 30 попередніх рядків охоплюють пропуск.

## Завдання 3. JSONB-таблиця для подій

**Що робимо:** SQL шаблону створює синтетичний event-log `t7_hw_events`: 50 000 подій п'яти типів, у кожного типу своя структура `payload`.

**Навіщо:** так виглядає типовий ML event-log. Спільні поля (`user_id`, `event_ts`, `event_type`) лежать у типізованих колонках, а деталі, які в кожного типу свої, зберігаються в `JSONB`. Усі значення задані формулами від `n`, тож при кожному запуску дані однакові.

In [69]:
%%sql
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

50000 rows affected.

5 rows affected.

event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


По одній події кожного типу, щоб побачити різні структури `payload`:

In [70]:
%%sql
SELECT DISTINCT ON (event_type) event_type, event_id, payload
FROM t7_hw_events
ORDER BY event_type, event_id;

5 rows affected.

event_type,event_id,payload
add_to_cart,2,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
checkout,3,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
click,1,"{'device': 'desktop', 'page_url': '/category/2', 'element_id': 'btn_2'}"
page_view,5,"{'url': '/product/6', 'device': 'tablet', 'referrer': 'direct'}"
recommendation_shown,4,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"


**Що вийшло:** `t7_hw_events` має 50 000 подій, по 10 000 кожного з п'яти типів. Структура `payload` у типів різна: `page_view` має `url`, `referrer` і `device`, `click` має `element_id`, `page_url` і `device`, `add_to_cart` має масив `tags`, `checkout` має `order_id`, `total` і `payment_method`, `recommendation_shown` має масив `recommendations`. Ключ `device` є лише у двох типах. jupysql показує JSONB як словник Python, тому лапки в output одинарні.

### 3.1. JSONB Query 1. Containment `@>`

**Що робимо:** шукаємо checkout-події, оплачені карткою. `payload @> '{"payment_method": "card"}'` питає: «чи містить документ цю пару ключ-значення».

**Навіщо:** це найчастіший фільтр по JSONB, і саме його прискорює GIN у Завданні 4.

In [71]:
%%sql
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

10 rows affected.

event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


**Що вийшло:** усі 10 рядків є checkout з `payment_method = card`. `event_id` іде з кроком 15 (3, 18, 33, …), бо тип події задає `n % 5`, а спосіб оплати `n % 3`. Карткою оплачена кожна третя checkout-подія.

### 3.2. JSONB Query 2. Key existence `?` та `?|`

**Що робимо:** `?` перевіряє, чи є в документі top-level ключ `tags`. `?|` перевіряє, чи є хоча б один ключ зі списку.

**Навіщо:** у різних типів подій різні ключі, тож за наявністю ключа видно, якого типу подія. Ці оператори бачать лише верхній рівень документа, а з індексів їх підтримує тільки GIN з default `jsonb_ops` (перевіримо в 4.5).

In [72]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

10 rows affected.

event_id,event_type,payload
2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [73]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

10 rows affected.

event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


**Що вийшло:** `? 'tags'` повертає лише `add_to_cart`, бо ключ `tags` є тільки в цьому типі. `?| ARRAY['experiment', 'model_version']` повертає лише `recommendation_shown`, де є обидва ключі. psycopg2 не сприйняв `?` як плейсхолдер, тому оператори працюють у `%%sql` без екранування.

### 3.3. JSONB Query 3. JSONPath

**Що робимо:** `jsonb_path_query` розгортає масив `recommendations` в окремі рядки. `jsonb_path_exists` з фільтром `? (@ > 300)` лишає checkout із сумою понад 300.

**Навіщо:** JSONPath дістає значення з вкладених масивів і перевіряє умову всередині документа, без ручного розбору JSON у Python.

In [74]:
%%sql
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

15 rows affected.

event_id,recommended_pid
4,5
4,6
4,7
9,10
9,11
9,12
14,15
14,16
14,17
19,20


In [75]:
%%sql
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;

10 rows affected.

event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


**Що вийшло:** `jsonb_path_query` дав 15 рядків з 5 подій: кожен масив `recommendations` з трьох id розгорнувся в три рядки, наприклад подія 4 дала 5, 6 і 7. `jsonb_path_exists` з `$.total ? (@ > 300)` лишив тільки checkout із сумою понад 300: перші 10 мають `total` від 303.00 до 348.00. Число порівнюється всередині документа, без касту в SQL.

## Завдання 4. GIN-index demo + EXPLAIN ANALYZE

**Що робимо:** порівнюємо плани JSONB-запитів до і після індексів: GIN `jsonb_path_ops` для `@>`, expression B-tree для `payload ->> 'device'` і додатково GIN `jsonb_ops` для `?`.

**Навіщо:** індекс корисний лише тоді, коли planner його реально бере. `EXPLAIN (ANALYZE, BUFFERS)` показує оператори плану, оцінку `cost`, фактичний час (`actual time`), кількість рядків і прочитані сторінки (`Buffers`). Функція `show_plan` друкує план звичайним текстом, рядок плану = рядок output, і не обрізає довгі рядки.

In [76]:
def show_plan(sql):
    """Виконує EXPLAIN і друкує план як текст, по рядку плану на рядок output."""
    with engine.connect() as conn:
        plan = conn.execute(text(sql)).scalars().all()
    print('\n'.join(plan))

### Step 1. Containment query без GIN-індексу

**Що робимо:** прибираємо індекси з кроків нижче (якщо Завдання 4 запускають повторно), оновлюємо статистику і дивимося план `@>`.

In [77]:
%%sql
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

SELECT indexname
FROM pg_indexes
WHERE tablename = 't7_hw_events'
ORDER BY indexname;

1 rows affected.

indexname
t7_hw_events_pkey


In [78]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""")

Aggregate  (cost=1756.58..1756.59 rows=1 width=8) (actual time=76.784..76.785 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=2633 width=0) (actual time=0.033..76.439 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=48 read=1
Planning Time: 0.595 ms
Execution Time: 76.812 ms


**Що вийшло:** план `Seq Scan on t7_hw_events`: PostgreSQL перебрав усі 50 000 рядків (1 000 сторінок, `Buffers: shared hit=1000`), 46 666 відкинув фільтром і лишив 3 334. Оцінка `cost=1758.19..1758.20`, фактичний час 20.521 ms. На таблиці є лише `t7_hw_events_pkey`, тож іншого шляху в planner немає.

### Step 2. GIN-індекс для containment / JSONPath workload

**Що робимо:** будуємо GIN з класом операторів `jsonb_path_ops`.

**Навіщо:** GIN працює як предметний покажчик у кінці книжки: для кожної пари шлях-значення він зберігає список рядків, де вона трапляється. `jsonb_path_ops` індексує лише хеші шляхів зі значеннями, тому він менший за default `jsonb_ops`, але підтримує тільки `@>`, `@?` і `@@`.

In [79]:
%%sql
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;

1 rows affected.

gin_index_size
1144 kB


### Step 3. Той самий containment query після GIN

**Що робимо:** повторюємо запит `payment_method = card` і додаємо другий containment `referrer = google`.

In [80]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""")

Aggregate  (cost=1091.75..1091.76 rows=1 width=8) (actual time=7.489..7.491 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=34.21..1083.53 rows=3288 width=0) (actual time=1.658..7.203 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..33.39 rows=3288 width=0) (actual time=1.490..1.491 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=33
Planning Time: 0.413 ms
Execution Time: 7.569 ms


In [81]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
""")

Aggregate  (cost=1086.29..1086.30 rows=1 width=8) (actual time=6.354..6.355 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.95..1078.67 rows=3048 width=0) (actual time=1.260..6.164 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..32.19 rows=3048 width=0) (actual time=1.068..1.069 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.169 ms
Execution Time: 6.417 ms


**Що вийшло:** GIN `jsonb_path_ops` займає 1144 kB. Після нього обидва containment-запити перейшли на `Bitmap Index Scan on idx_t7_events_payload_gin_pathops` + `Bitmap Heap Scan`. Для `card` `cost` впав з 1758.20 до 1121.68, а час з 20.521 до 2.305 ms, майже в 9 разів. Для `referrer = google` (2 500 рядків) `cost=1071.03` і 1.635 ms.

Сам індекс прочитав лише 4 сторінки, але `Heap Blocks: exact=1000`: потрібні рядки розкидані по всіх 1 000 сторінках таблиці, бо типи подій чергуються через `n % 5`. Тому heap читається повністю, і `Buffers` майже не змінились (1 004 проти 1 000). Час виграно на тому, що `@>` перевіряється лише для 3 334 кандидатів з індексу (`Recheck Cond`), а не для 50 000 рядків.

### Step 4. Коли GIN не допомагає: path-specific text equality

**Що робимо:** запит `payload ->> 'device' = 'mobile'` спершу з GIN на всій колонці, потім з expression B-tree index на виразі `payload ->> 'device'`.

**Навіщо:** `->>` повертає текст, і `=` порівнює цей текст. GIN не вміє відповідати на такий вираз: він індексує документ, а не результат `->>`. Expression index схожий на окремий алфавітний список саме за одним полем.

In [82]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
""")

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=12.846..12.848 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.015..11.820 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.065 ms
Execution Time: 12.871 ms


In [83]:
%%sql
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

SELECT indexname
FROM pg_indexes
WHERE tablename = 't7_hw_events'
ORDER BY indexname;

3 rows affected.

indexname
idx_t7_events_device_btree
idx_t7_events_payload_gin_pathops
t7_hw_events_pkey


In [84]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
""")

Aggregate  (cost=1242.55..1242.56 rows=1 width=8) (actual time=4.651..4.653 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.82..1221.73 rows=8327 width=0) (actual time=0.604..4.106 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.74 rows=8327 width=0) (actual time=0.460..0.461 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=26 read=1
Planning Time: 0.287 ms
Execution Time: 4.712 ms


**Що вийшло:** без expression index запит іде через `Seq Scan`: 1 000 сторінок, 41 667 рядків відкинуто, 8 333 лишилось, 4.317 ms. GIN на `payload` у таблиці вже є, але planner його не бере: GIN не відповідає на вираз `payload ->> 'device' = 'mobile'`. Оцінка теж погана: planner чекав `rows=250` замість 8 333, бо статистики для виразу ще немає.

Після `CREATE INDEX` на `(payload ->> 'device')` і `ANALYZE` план став `Bitmap Index Scan on idx_t7_events_device_btree` + `Bitmap Heap Scan`. Оцінка `rows=8357` вже близька до факту, `cost` впав з 1750.63 до 1243.31, час з 4.317 до 1.428 ms. B-tree прочитав 9 сторінок.

### 4.5. Додатково: `?` при `jsonb_path_ops` і при `jsonb_ops`

**Що робимо:** план `payload ? 'tags'`, поки є лише GIN `jsonb_path_ops`. Потім створюємо GIN з default `jsonb_ops` (назва `idx_t7_events_payload_gin_ops` уже є в `DROP` кроку 1) і повторюємо.

**Навіщо:** перевірити на плані примітку з умови: `?`, `?|`, `?&` індексуються тільки через `jsonb_ops`.

In [85]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags';
""")

Aggregate  (cost=1649.77..1649.78 rows=1 width=8) (actual time=10.573..10.574 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=9908 width=0) (actual time=0.013..9.972 rows=10000 loops=1)
        Filter: (payload ? 'tags'::text)
        Rows Removed by Filter: 40000
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.186 ms
Execution Time: 10.597 ms


In [86]:
%%sql
CREATE INDEX idx_t7_events_payload_gin_ops
    ON t7_hw_events USING GIN (payload);  -- default operator class: jsonb_ops

ANALYZE t7_hw_events;

SELECT indexname
FROM pg_indexes
WHERE tablename = 't7_hw_events'
ORDER BY indexname;

4 rows affected.

indexname
idx_t7_events_device_btree
idx_t7_events_payload_gin_ops
idx_t7_events_payload_gin_pathops
t7_hw_events_pkey


In [87]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags';
""")

Aggregate  (cost=1228.59..1228.60 rows=1 width=8) (actual time=7.018..7.019 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=77.93..1203.48 rows=10044 width=0) (actual time=1.585..6.391 rows=10000 loops=1)
        Recheck Cond: (payload ? 'tags'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_ops  (cost=0.00..75.42 rows=10044 width=0) (actual time=1.326..1.326 rows=10000 loops=1)
              Index Cond: (payload ? 'tags'::text)
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=32
Planning Time: 0.328 ms
Execution Time: 7.085 ms


**Що вийшло:** поки є лише `jsonb_path_ops`, `payload ? 'tags'` іде через `Seq Scan` (`cost=1650.12`, 3.597 ms, 40 000 рядків відкинуто): цей клас операторів не підтримує `?`. З GIN `jsonb_ops` план став `Bitmap Index Scan on idx_t7_events_payload_gin_ops`, `cost=1229.99`, 2.579 ms. Ключ `tags` є в 10 000 рядках, тобто у 20 % таблиці, тому виграш у часі менший, ніж для `@>`.

### 4.6. Розміри таблиці та індексів

**Що робимо:** порівнюємо розмір таблиці й кожного індексу (`pg_relation_size`).

**Навіщо:** кожен індекс є окремою структурою на диску, яку треба оновлювати при кожному `INSERT`/`UPDATE`. Розмір показує, скільки коштує індекс.

In [88]:
%%sql
SELECT c.relname AS object_name,
       am.amname AS access_method,
       pg_size_pretty(pg_relation_size(c.oid)) AS size,
       pg_relation_size(c.oid) AS size_bytes
FROM pg_class AS c
LEFT JOIN pg_am AS am ON am.oid = c.relam
WHERE c.relname IN ('t7_hw_events',
                    't7_hw_events_pkey',
                    'idx_t7_events_payload_gin_pathops',
                    'idx_t7_events_payload_gin_ops',
                    'idx_t7_events_device_btree')
ORDER BY size_bytes DESC;

5 rows affected.

object_name,access_method,size,size_bytes
t7_hw_events,heap,8000 kB,8192000
idx_t7_events_payload_gin_ops,gin,1696 kB,1736704
idx_t7_events_payload_gin_pathops,gin,1144 kB,1171456
t7_hw_events_pkey,btree,1112 kB,1138688
idx_t7_events_device_btree,btree,360 kB,368640


**Що вийшло:** таблиця займає 8000 kB. GIN `jsonb_ops` має 1696 kB, `jsonb_path_ops` 1144 kB, приблизно на третину менше. PK `t7_hw_events_pkey` займає 1112 kB, а expression B-tree на одне поле `device` лише 360 kB. Разом чотири індекси важать 4312 kB, більше за половину самої таблиці, і кожен `INSERT` оновлює їх усі.

## Завдання 5. JSONB vs normalized

### 5.1. Нормалізована таблиця для checkout-подій

**Що робимо:** SQL шаблону переносить 10 000 checkout-подій у таблицю з окремою типізованою колонкою на кожне поле `payload`, з `CHECK` і B-tree індексом на `payment_method`.

**Навіщо:** це альтернатива «усе в JSONB». Типи й `CHECK` перевіряють дані при записі, а запит читає лише потрібні рядки.

In [89]:
%%sql
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;

10000 rows affected.

1 rows affected.

n_checkouts
10000


### 5.2. Один запит двома способами

**Що робимо:** середній `total` і кількість checkout за способом оплати, топ-3. Спершу перевіряємо, що обидві версії дають однакову відповідь, потім порівнюємо плани.

**Навіщо:** порівнювати швидкість має сенс лише для запитів з однаковим результатом.

In [90]:
%%sql
SELECT 'jsonb' AS source,
       payload ->> 'payment_method' AS payment_method,
       ROUND(AVG((payload ->> 'total')::NUMERIC), 2) AS avg_total,
       COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
UNION ALL
SELECT 'normalized',
       payment_method,
       ROUND(AVG(total), 2),
       COUNT(*)
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY payment_method, source;

6 rows affected.

source,payment_method,avg_total,n
jsonb,card,300.50,3334
normalized,card,300.50,3334
jsonb,crypto,300.48,3333
normalized,crypto,300.48,3333
jsonb,paypal,300.52,3333
normalized,paypal,300.52,3333


JSONB-версія:

In [91]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
""")

Limit  (cost=1952.93..1952.94 rows=3 width=72) (actual time=15.837..15.839 rows=3 loops=1)
  Buffers: shared hit=1003
  ->  Sort  (cost=1952.93..1966.72 rows=5513 width=72) (actual time=15.836..15.837 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1003
        ->  HashAggregate  (cost=1798.99..1881.68 rows=5513 width=72) (actual time=15.790..15.816 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.86 rows=9942 width=132) (actual time=0.010..10.347 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=41
Planning Time: 0.172 ms
Execution Time: 16.021 ms


Normalized-версія:

In [92]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=6.238..6.240 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=6.236..6.237 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=6.225..6.228 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.013..1.574 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=34
Planning Time: 0.256 ms
Execution Time: 6.276 ms


**Що вийшло:** обидві версії дають однакову відповідь: card 300.50 (3 334 checkout), crypto 300.48 (3 333), paypal 300.52 (3 333). Плани різні.

- JSONB: `Seq Scan on t7_hw_events` читає всі 1 000 сторінок і відкидає 40 000 подій інших типів, далі `HashAggregate` дістає `payload ->> 'payment_method'` і `payload ->> 'total'` з 10 000 документів. `cost=1953.77`, 5.866 ms. Planner очікував 5 527 груп замість 3, бо статистики для виразу немає.
- Normalized: `Seq Scan on t7_hw_checkouts_norm` читає 94 сторінки, де лежать лише checkout-и з уже типізованими колонками. `cost=269.07`, 1.768 ms, оцінка груп точна (3).

Normalized-версія читає приблизно в 10 разів менше сторінок і в цьому середовищі працює в 3.3 раза швидше. Індекс `idx_t7_checkouts_norm_pm` planner не взяв: запит агрегує всю таблицю, і послідовно прочитати 94 сторінки дешевше.

### 5.3. Bonus: MongoDB-equivalent як pseudocode

Той самий запит (середній `total` checkout за способом оплати, топ-3) як aggregation pipeline. Це pseudocode, він не виконується в PostgreSQL.

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

Кроки відповідають SQL один до одного: `$match` = `WHERE`, `$group` = `GROUP BY` з `AVG` і `COUNT`, `$sort` + `$limit` = `ORDER BY ... LIMIT 3`. Чим підходи відрізняються:

- PostgreSQL JSONB зручний, коли поряд з документами потрібні SQL, `JOIN` з нормалізованими таблицями і транзакції: подію можна записати в одній транзакції із замовленням і приєднати до таблиці користувачів.
- MongoDB aggregation pipeline природний для document-first навантаження, де головна одиниця даних це документ, а схема часто змінюється.
- Сирий clickstream великого обсягу часто краще тримати в document store або data lake, а в PostgreSQL переносити ETL-ом лише потрібне.
- Curated ML features зручніше тримати в typed SQL-таблицях: типи й `CHECK` ловлять помилки при записі, а запити й індекси прості, як у `t7_hw_checkouts_norm`.

## Reflection

1. До GIN запит `payload @> {"payment_method": "card"}` ішов через `Seq Scan`: `cost` 1758.20, 20.521 ms, 46 666 рядків відкинуто. Після GIN `jsonb_path_ops` план став `Bitmap Index Scan` + `Bitmap Heap Scan`: `cost` 1121.68, 2.305 ms. Сторінок прочитано майже стільки ж (1 004 проти 1 000), бо 3 334 потрібні рядки розкидані по всій таблиці. Виграш дає те, що `@>` перевіряється лише для кандидатів з індексу.

2. GIN зберігає вміст документа: ключі й значення (`jsonb_ops`) або хеші шляхів зі значеннями (`jsonb_path_ops`). `@>` питає саме про такий вміст. Умова `payload ->> 'device' = 'mobile'` порівнює текст, який повертає вираз, а результатів виразу GIN не зберігає. Тому planner узяв `Seq Scan` (4.317 ms, оцінка 250 рядків замість 8 333). Expression B-tree зберігає саме `payload ->> 'device'`: 360 kB, `Bitmap Index Scan` і 1.428 ms.

3. GIN пришвидшує читання для `@>`, `?` і JSONPath-оператора `@?`, але займає місце (`jsonb_ops` 1696 kB, `jsonb_path_ops` 1144 kB при таблиці 8000 kB) і оновлюється при кожному `INSERT` та зміні `payload`. Один документ дає в індексі багато записів, тож для event-log, куди пишуть постійно, запис стає дорожчим. Індексувати варто лише запити, які справді часті.

4. JSONB я взяв би для сирого event-log, де в кожного типу своя схема і поля додаються часто. Нормалізовану таблицю взяв би для curated ML-ознак і полів, за якими постійно фільтрують та агрегують. На однаковому запиті normalized-версія прочитала 94 сторінки замість 1 000 і виконалась за 1.768 ms проти 5.866 ms, а `CHECK` ловить помилки ще під час запису.

5. У JSONB-підході `refund` означає лише нові рядки з `event_type = 'refund'` і своїм `payload` (`order_id`, `amount`, `reason`), без DDL. Треба оновити код, який читає події, і за потреби додати expression index на `payload ->> 'order_id'`. У normalized-підході потрібна міграція: `CREATE TABLE t7_hw_refunds_norm` з типами й `CHECK`, `INSERT ... SELECT` для історичних подій, індекс на `order_id` і зміни в ETL.